## Agent的基本使用

### 1、创建Agent

In [1]:
from langchain_classic.chains.hyde.prompts import web_search
# model
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash", # 模型名称
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

In [2]:
# tool

# 自定义工具
from langchain_core.tools import tool
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取指定城市的信息

    Args:
         city : 具体的城市

    Returns:
        返回城市的天气信息
    """
    return city + "晴天，温度15°C"

# 使用langchain工具
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results=1,
    topic="general",
)


In [3]:
# agent
from langchain.agents import create_agent

agent = create_agent(
    name="test_agent", # 一般用于Multi-Agent场景
    model = model,
    tools = [get_weather, tavily_search_tool],
    # system_prompt="你是一个话少的助手"
)

rprint(agent)
rprint(type(agent))

<langgraph.graph.state.CompiledStateGraph object at 0x00000268E56D5310>

<class 'langgraph.graph.state.CompiledStateGraph'>

### 2、agent调用

In [4]:
# 入参类型为字典，输出类型为字典
response = agent.invoke({
    "messages": [
        {"role": "system", "content": "你是一个话少的助手"},
        {"role": "user", "content": "北京的天气如何"},
        {"role": "user", "content": "2026年巴威台风对苏州的影响如何"}
    ]
})

rprint(response)

{
    'messages': [
        SystemMessage(
            content='你是一个话少的助手',
            additional_kwargs={},
            response_metadata={},
            id='55a563f0-c520-4365-bbcf-b3a8335e3644'
        ),
        HumanMessage(
            content='北京的天气如何',
            additional_kwargs={},
            response_metadata={},
            id='3a379626-71a8-464e-a475-b68962ae60d6'
        ),
        HumanMessage(
            content='2026年巴威台风对苏州的影响如何',
            additional_kwargs={},
            response_metadata={},
            id='fe8871ef-e93b-4c98-8479-7d4d3ac21508'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 84,
                    'prompt_tokens': 1882,
                    'total_tokens': 1966,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    },
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 1882
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '7356b772-518c-490f-9c39-c7dc2479a31b',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='test_agent',
            id='lc_run--01a0385d-f5d1-7b12-9a9b-5f23e6957520-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_f5Nh3nDKG3OtEpSiUxUp3358',
                    'type': 'tool_call'
                },
                {
                    'name': 'tavily_search',
                    'args': {'query': '2026年巴威台风 苏州影响'},
                    'id': 'call_01_AhVsgj5AfHmXyhrqhCiM0186',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1882,
                'output_tokens': 84,
                'total_tokens': 1966,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='北京晴天，温度15°C',
            name='get_weather',
            id='97d142e3-3324-4d79-b25e-316d6441c25c',
            tool_call_id='call_00_f5Nh3nDKG3OtEpSiUxUp3358'
        ),
        ToolMessage(
            content='{"query": "2026年巴威台风 苏州影响", "follow_up_questions": null, "answer": null, "images": 
[], "results": [{"url": "https://www.suzhou.gov.cn/szsrmzf/mszx/202607/4c33cdb009da453dbfd7b9e75b5856ed.shtml", 
"title": "五问超强台风“巴威” - 苏州市人民政府", "content": "## 当前位置： 首页 > 新闻中心 > 便民公告\\n\\n# 
五问超强台风“巴威”\\n\\n## 
\\n\\n今年第9号台风“巴威”（超强台风级）的中心今天（8日）8点位于台湾基隆市东偏南方向约1580公里的洋面上（北纬16.9度、
东经134.1度），中心附近最大风力17级以上（62米/秒）。\\n\\n受其影响，台湾、浙江、福建将率先迎来强降雨；随着台风深入
内陆，雨带还将继续北推，先后波及长江中下游，甚至跨过长江影响到华北、黄淮等地。\\n\\n“巴威”的强度、路径、可能的影响
如何？应怎样防御？围绕公众关心的这些问题，中央气象台台风首席预报员向纯怡进行了解答。\\n\\n一问：“巴威”为何这么猛？\
\n\\n“巴威”强度大、能量足，且维持超强台风的时间较长。未来靠近台湾以东洋面前，都将维持超强台风等级。从气象卫星云图上
可以清晰看到，“巴威”台风眼清晰、浑圆，中心密闭云区密实。\\n\\n风云四号B星真彩色云图连续监测台风动画（2026年7月7日4:
00至11:00）\\n\\n“‘巴威’如此猛烈，是因为它具备了成为超强台风的所有有利条件。”向纯怡解释——\\n\\n首先，台风行进路径上
热带洋面海温高，为台风发展、能量补给提供了有利的热力条件。 [...] 
其次，持续稳定的水汽不断向台风中心输送能量，使台风环流始终保持旺盛状态。\\n\\n此外，当前高空辐散条件维持台风内核结
构稳定，利于强度维持和发展。\\n\\n二问：“巴威”路径怎么走？\\n\\n预计，“巴威”将以每小时15-20公里的速度向偏西转西北方
向移动，逐渐向台湾岛东北部沿海靠近，强度先维持，9日开始逐渐减弱。\\n\\n“巴威”可能于10日夜间至11日登陆或擦过台湾岛北
部沿海，然后于11日夜间至12日在浙闽交界附近沿海登陆；也可能在台湾岛以东洋面北上，直接登陆浙江沿海。\\n\\n三问：“巴威
”登陆后影响几何？\\n\\n预计，从9日开始，“巴威”将开始给我国东部海域及东南沿海带来大范围的强风雨过程，东海大部、黄海
南部、台湾以东洋面、钓鱼岛附近海域、台湾岛北部、台湾海峡、巴士海峡、华东沿海将先后受到“巴威”的强风影响。\\n\\n9日至
15日，华东、华中、华北等地将先后出现强降水。\\n\\n需要特别关注的是，“巴威”登陆后，其残余环流、外围云系仍会深入内陆
，给我国内